# End-to-End CatBoost Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with CatBoost gradient-boosted models.
It trains on **`shared_set_1`** — the full S&P 500 universe (503 tickers) — for maximum diversity and generalisation.

It covers the full workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer a custom notebook-local feature
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. train three CatBoost regressors (return, alpha, volatility)
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow as run `Felix_Run14_regime_smooth_cap02`


## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    backtest_weights,
    write_backtest_artifacts,
    log_model_submission,
    validate_feature_frame,
)

print('Imports loaded successfully.')


In [ ]:
# Resolve repo_root robustly — always anchor to this notebook's location.
repo_root = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path().resolve()
# If the above still resolves incorrectly, uncomment and set explicitly:
repo_root = Path(r'C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib')
print('repo_root:', repo_root)
dataset_name = 'shared_set_1'
model_name = 'catboost_sp500'
run_name = 'Felix_Run14_regime_smooth_cap02'

# Concentration-aware allocator controls.
# Diagnostics from Run 13 showed lower volatility/drawdown but high turnover and
# high market correlation. Run 14 keeps the model fixed and improves only sizing:
# beta/realized-vol penalties, high-risk regime flattening, and weight smoothing.
MAX_WEIGHT_CAP = 0.02
MIN_ACTIVE_NAMES = 75
SCORE_SOFTMAX_TEMPERATURE = 0.35
HIGH_RISK_SOFTMAX_TEMPERATURE = 0.60
EQUAL_WEIGHT_BLEND = 0.12
HIGH_RISK_EQUAL_WEIGHT_BLEND = 0.30
BETA_PENALTY = 0.20
HIGH_RISK_BETA_PENALTY = 0.40
REALIZED_VOL_PENALTY = 0.15
HIGH_RISK_REALIZED_VOL_PENALTY = 0.30
DOWNSIDE_VOL_PENALTY = 0.10
WEIGHT_SMOOTHING = 0.60
RISK_VIX_LEVEL = 25.0
RISK_VIX_CHANGE_5D = 0.15
RISK_TLT_MOMENTUM_20D = -0.02
# ── Prediction horizon ───────────────────────────────────────────────────────
# Options: 5 (weekly), 10 (bi-weekly), 21 (monthly)
# Longer horizons = less noise but slower signal; shorter = more trades, higher costs
horizon = 10  # Changed to 10d (bi-weekly) — better signal-to-noise than 21d or 5d

# ── Rebalancing frequency ────────────────────────────────────────────────────
# Options: 1 (daily), 5 (weekly), 21 (monthly)
# Should match or be shorter than horizon for consistency
rebalance_every_n_days = 10  # bi-weekly rebalancing — matches prediction horizon
output_dir = repo_root / 'runs' / 'catboost_sp500_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

## 2. Add Built-In Toolkit Features

We start with a full built-in feature set.

**Important — leak prevention:** Features are built from the full price history
(all rolling features look backward only, so no future data is used).
However to prevent any subtle leakage from the `dropna()` step interacting with
split boundaries, we split the raw price data first and build features
separately per split, then reassemble. This ensures val/test feature distributions
are never influenced by future price history.


In [ ]:
# ── Split-before-feature-engineering (leak prevention) ──────────────────────
splits_cfg = split_dates(dataset_name, repo_root=repo_root)

train_cutoff = pd.Timestamp("2020-12-31")
val_end      = pd.Timestamp("2021-12-31")
test_start   = pd.Timestamp("2022-01-03")
train_start  = pd.Timestamp("2014-01-02")

# ── Embargo gap: 10 trading days between train end and val start ──────────────
# Prevents target leakage from overlapping forward windows.
# Idea borrowed from Brian's LightGBM model.
EMBARGO_DAYS = 10

# ── Richer base feature set (borrowed from Brian and Hannah) ──────────────────
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_5d',
    'momentum_20d',
    'momentum_60d',
    'momentum_120d',          # NEW: longer-term momentum
    'downside_vol_20d',       # NEW: downside-only volatility
    'upside_vol_20d',         # NEW: upside-only volatility
    'distance_to_20d_high',   # NEW: how far below 20d peak
    'distance_to_60d_high',   # NEW: how far below 60d peak
    'rsi_14',
    'macd_hist',              # NEW: trend acceleration
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'dollar_volume_ratio_20d',# NEW: liquidity proxy
    'excess_return_5d_vs_spy',
    'excess_return_20d_vs_spy',
    'excess_return_60d_vs_spy',
    'relative_momentum_20d_vs_spy',  # NEW: relative strength
    'beta_20d_spy',
    'beta_60d_spy',           # NEW: longer-term beta
    'intraday_range',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
print(f'Feature count: {len(base_feature_names)} (expanded from 12)')
display(base_features.head())


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom. The toolkit supplies the base feature set;
everything below is notebook-local and experimental.

We add three categories of custom features:

**Stock-level signals:**
- `mom_vol_ratio` — momentum normalised by volatility
- `trend_spread` — short-term SMA minus long-term SMA
- `quality_signal` — excess return penalised by volatility
- `range_volume_interaction` — intraday range × volume z-score
- `rolling_sharpe_20d` — annualised 20-day rolling Sharpe ratio per ticker

**Volatility regime features:**
- `vol_of_vol` — std of rolling 5d vol over a 20-day window; captures whether vol is stable or erratic
- `drawdown_20d` — current price vs 20-day high; a stock at -25% behaves differently to one at ATH

**Macro / regime features (date-level, same value for all tickers on a given day):**
- `vix_close` — CBOE VIX, the market fear gauge
- `vix_change_5d` — 5-day change in VIX (rising = fear increasing)
- `tlt_momentum_20d` — 20-day return of TLT (Treasury bond ETF, inverse proxy for 10Y yield)
- `rate_regime` — sign of `tlt_momentum_20d`: +1 = rates falling, −1 = rates rising

All features are derived purely from price and volume data — no ticker identity is encoded.
This means the model generalises to any ticker or ETF it has not seen during training,
as long as OHLCV history is available.


In [ ]:
frame = base_features.copy()
eps = 1e-6

# ── Stock-level custom features ──────────────────────────────────────────────
frame['mom_vol_ratio']             = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread']              = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal']            = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction']  = frame['intraday_range'] * frame['volume_zscore_20d']

# ── Volatility regime features ────────────────────────────────────────────────
daily_returns = (
    prices.sort_values(['ticker', 'date'])
    .assign(daily_ret=lambda df: df.groupby('ticker')['close'].pct_change())
)

# rolling_sharpe_20d
rolling_sharpe = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(20).mean() / (s.rolling(20).std(ddof=0) + eps))
)
daily_returns['rolling_sharpe_20d'] = rolling_sharpe * (252 ** 0.5)
sharpe_lookup = daily_returns[['date', 'ticker', 'rolling_sharpe_20d']]
frame = frame.merge(sharpe_lookup, on=['date', 'ticker'], how='left')

# vol_of_vol
vol_of_vol = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(5).std().rolling(20).std())
)
daily_returns['vol_of_vol'] = vol_of_vol
vov_lookup = daily_returns[['date', 'ticker', 'vol_of_vol']]
frame = frame.merge(vov_lookup, on=['date', 'ticker'], how='left')

# drawdown_20d
drawdown_20d = (
    prices.sort_values(['ticker', 'date'])
    .groupby('ticker')['close']
    .transform(lambda s: s / s.rolling(20).max() - 1)
)
drawdown_lookup = (
    prices.sort_values(['ticker', 'date'])[['date', 'ticker']]
    .assign(drawdown_20d=drawdown_20d.values)
)
frame = frame.merge(drawdown_lookup, on=['date', 'ticker'], how='left')

# ── Macro / regime features ───────────────────────────────────────────────────
import yfinance as yf
_start = '2014-01-01'
_end   = '2025-12-31'

def _fetch_close(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    close = df['Close']
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close

vix_raw = _fetch_close("^VIX", _start, _end).rename("vix_close")
tlt_raw = _fetch_close("TLT",  _start, _end).rename("tlt_close")

macro = (
    pd.concat([vix_raw, tlt_raw], axis=1)
    .reset_index()
    .rename(columns={'Date': 'date', 'Datetime': 'date', 'index': 'date'})
    .assign(date=lambda df: pd.to_datetime(df['date']).dt.normalize())
)
macro['vix_change_5d']    = macro['vix_close'].pct_change(5)
macro['tlt_momentum_20d'] = macro['tlt_close'].pct_change(20)
macro['rate_regime']      = macro['tlt_momentum_20d'].apply(lambda x: 1.0 if x >= 0 else -1.0)
macro_feature_names = ['vix_close', 'vix_change_5d', 'tlt_momentum_20d', 'rate_regime']
macro = macro[['date'] + macro_feature_names].dropna()
frame['date'] = pd.to_datetime(frame['date']).dt.normalize()
frame = frame.merge(macro, on='date', how='left')
print('Macro features merged.')

# ── Regime & categorical features ────────────────────────────────────────────
frame['vix_regime'] = pd.cut(
    frame['vix_close'],
    bins=[0, 15, 20, 30, 100],
    labels=[0, 1, 2, 3]
).astype(float)

frame['momentum_regime'] = pd.cut(
    frame['momentum_20d'],
    bins=[-np.inf, -0.05, 0, 0.05, np.inf],
    labels=[0, 1, 2, 3]
).astype(float)

eps2 = 1e-6
daily_ret_std5  = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(5).std())
daily_ret_std20 = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(20).std())
vol_expansion = daily_ret_std5 / (daily_ret_std20 + eps2)
vol_exp_lookup = (
    daily_returns[['date', 'ticker']]
    .assign(vol_expansion_ratio=vol_expansion.values)
)
frame = frame.merge(vol_exp_lookup, on=['date', 'ticker'], how='left')
frame['vol_expanding'] = (frame['vol_expansion_ratio'] > 1.2).astype(float)

print('Regime and categorical features added.')

# ── Cross-sectional rank features (computed AFTER split to avoid leakage) ───
# These rank each stock relative to all others on the SAME date.
# IMPORTANT: ranks are computed separately on train, val, test to avoid
# future dates influencing the rank distribution of past dates.
# We add them to the frame now as NaN placeholders — actual values filled in cell 15.
frame['xs_return_rank']   = np.nan  # rank of return_20d across tickers per date
frame['xs_momentum_rank'] = np.nan  # rank of momentum_20d across tickers per date
frame['xs_sharpe_rank']   = np.nan  # rank of rolling_sharpe_20d across tickers per date
frame['xs_quality_rank']  = np.nan  # rank of quality_signal across tickers per date
print("Cross-sectional rank placeholders added — will be filled per split in cell 15.")

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
    'rolling_sharpe_20d',
    'vol_of_vol',
    'drawdown_20d',
    'vix_close',
    'vix_change_5d',
    'tlt_momentum_20d',
    'rate_regime',
    'xs_return_rank',
    'xs_momentum_rank',
    'xs_sharpe_rank',
    'xs_quality_rank',
    'vix_regime',
    'momentum_regime',
    'vol_expansion_ratio',
    'vol_expanding',
]

all_feature_names = base_feature_names + custom_feature_names
print(f'Total features: {len(all_feature_names)} ({len(base_feature_names)} base + {len(custom_feature_names)} custom)')
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [ ]:
# ── Build targets for multiple horizons ──────────────────────────────────────
return_targets_5d  = make_forward_return_target(prices, horizon=5)
return_targets_10d = make_forward_return_target(prices, horizon=10)
return_targets_21d = make_forward_return_target(prices, horizon=21)

alpha_targets_5d   = make_forward_alpha_target(prices, horizon=5)
alpha_targets_10d  = make_forward_alpha_target(prices, horizon=10)
alpha_targets_21d  = make_forward_alpha_target(prices, horizon=21)

vol_targets_5d     = make_forward_realized_vol_target(prices, window=5)
vol_targets_10d    = make_forward_realized_vol_target(prices, window=10)
vol_targets_21d    = make_forward_realized_vol_target(prices, window=21)

# Merge all target horizons into the feature frame
target_frame = frame.copy()
for df in [
    return_targets_5d, return_targets_10d, return_targets_21d,
    alpha_targets_5d,  alpha_targets_10d,  alpha_targets_21d,
    vol_targets_5d,    vol_targets_10d,    vol_targets_21d,
]:
    target_frame = target_frame.merge(df, on=['date', 'ticker'], how='left')

# ── Drop NaNs excluding rank columns ─────────────────────────────────────────
# xs_*_rank columns are NaN placeholders at this stage — they get filled
# per split in cell 15 to prevent leakage. Exclude them from dropna().
rank_cols = [c for c in target_frame.columns if c.startswith("xs_")]
cols_to_check = [c for c in target_frame.columns if c not in rank_cols]

target_frame = (
    target_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=cols_to_check)
    .reset_index(drop=True)
)

# ── Active target columns ─────────────────────────────────────────────────────
return_target_col = f"forward_return_{horizon}d"
alpha_target_col  = f"forward_alpha_{horizon}d_vs_spy"
vol_target_col    = f"forward_realized_vol_{horizon}d"

print(f"Active horizon: {horizon}d")
print(f"Return target:  {return_target_col}")
print(f"Alpha target:   {alpha_target_col}")
print(f"Vol target:     {vol_target_col}")
print("Modeling frame shape after dropping nulls:", target_frame.shape)
print("Rank cols still NaN (expected):", target_frame[rank_cols].isna().all().all())
display(target_frame.head())


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. **Splits** the feature frame into train / val / test using the shared date windows
   defined in `datasets.toml` — so every model in the repo is evaluated on the same periods.

2. **Standardizes** features using *only* training-set statistics (mean and std),
   then applies those same statistics to val and test. This prevents data leakage.

3. **Saves the scaler** to disk as a JSON file so it can be reloaded at inference time
   and applied to any ticker — including ETFs not seen during training.
   Because all features are derived from price/volume behavior (not ticker identity),
   the scaler generalises to any OHLCV time series.


In [ ]:
import json as _json

# ── Cross-sectional rank function (defined first so it can be called below) ──
def add_xs_ranks(df):
    """Add cross-sectional percentile ranks per date within this split only.
    Computed per split to prevent leakage across train/val/test boundaries.
    """
    df = df.copy()
    for src_col, rank_col in [
        ('return_20d',         'xs_return_rank'),
        ('momentum_20d',       'xs_momentum_rank'),
        ('rolling_sharpe_20d', 'xs_sharpe_rank'),
        ('quality_signal',     'xs_quality_rank'),
    ]:
        if src_col in df.columns:
            df[rank_col] = df.groupby('date')[src_col].rank(pct=True)
    return df

# ── Override train cutoff to 2020-12-31 ──────────────────────────────────────
# Ensure date column is datetime before filtering
target_frame["date"] = pd.to_datetime(target_frame["date"])

train = target_frame[
    (target_frame["date"] >= pd.Timestamp("2014-01-02")) &
    (target_frame["date"] <= pd.Timestamp("2020-12-31"))
].copy()
# Embargo: skip the first EMBARGO_DAYS trading days after train cutoff
embargo_end = pd.Timestamp("2020-12-31") + pd.Timedelta(days=EMBARGO_DAYS * 2)
val = target_frame[
    (target_frame["date"] >= embargo_end) &
    (target_frame["date"] <= pd.Timestamp("2021-12-31"))
].copy()
test = target_frame[
    (target_frame["date"] >= pd.Timestamp("2022-01-03")) &
    (target_frame["date"] <= pd.Timestamp("2025-12-31"))
].copy()

# Apply ranks per split (no leakage)
train = add_xs_ranks(train)
val   = add_xs_ranks(val)
test  = add_xs_ranks(test)
print("Cross-sectional ranks computed per split.")
print("Overridden splits:")
print("Train rows:", len(train), "| cutoff: 2020-12-31")
print("Val rows:  ", len(val))
print("Test rows: ", len(test))

# ── Rebalancing frequency subsampling ────────────────────────────────────────
def get_rebalance_dates(df, every_n_days):
    """Return every Nth unique date from a frame."""
    dates = sorted(df["date"].unique())
    return dates[::every_n_days]

rebalance_dates = get_rebalance_dates(test, rebalance_every_n_days)
test_rebalance  = test[test["date"].isin(rebalance_dates)].copy()
print(f"\nRebalance every {rebalance_every_n_days} days")
print(f"Full test rows: {len(test)} → Rebalance test rows: {len(test_rebalance)}")
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# ── Build scaler from training data only (no leakage) ────────────────────────
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

output_dir.mkdir(parents=True, exist_ok=True)
scaler_path = output_dir / "feature_scaler.json"
scaler_dict = {
    "feature_names": all_feature_names,
    "means":         train_means.to_dict(),
    "stds":          train_stds.to_dict(),
    "dataset":       dataset_name,
    "horizon":       horizon,
    "rebalance_every_n_days": rebalance_every_n_days,
}
with open(scaler_path, "w") as f:
    _json.dump(scaler_dict, f, indent=2)
print(f"Scaler saved to: {scaler_path}")

# ── Standardize splits ───────────────────────────────────────────────────────
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.fillna(0.0).to_numpy(dtype=float)

X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test_rebalance)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test_rebalance[return_target_col].to_numpy(dtype=float)

y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)

print("X_train shape:", X_train.shape)
print("X_test shape (rebalanced):", X_test.shape)
print("Feature count:", X_train.shape[1])
print("NaN values after standardization:", np.isnan(X_train).sum())


In [ ]:
# ── How to reload the scaler for inference on unseen tickers / ETFs ─────────
# This block is for reference — it shows how to apply the saved scaler
# to a completely new dataframe (e.g. ETF test data not in shared_set_2).
#
# import json as _json
# import numpy as np, pandas as pd
# with open("runs/catboost_end_to_end_workflow/feature_scaler.json") as f:
#     scaler = _json.load(f)
#
# saved_means = pd.Series(scaler["means"])
# saved_stds  = pd.Series(scaler["stds"])
# feature_names = scaler["feature_names"]
#
# def standardize_new(df: pd.DataFrame) -> np.ndarray:
#     return ((df[feature_names] - saved_means) / saved_stds).fillna(0.0).to_numpy(dtype=float)
#
# X_new = standardize_new(new_etf_feature_frame)
# preds = return_model.predict(X_new)  # works on any ticker

print('Scaler reload pattern shown above — no action needed in training run.')


## 6. Train CatBoost Models

Two CatBoost models:

1. **Joint return + alpha model** with `MultiRMSE` — predicts both targets simultaneously.

2. **Vol rank model** — predicts cross-sectional vol rank (0–1).

Predictions feed into a custom risk-aware portfolio builder with:
- **Top-N selection** (top 100 stocks per rebalance)
- **Risk-aware weighting** — `alpha_score / vol^0.5`
- **Per-stock weight cap** — max 4% per name (prevents concentration)
- **Turnover blend** — 60% new signal + 40% previous weights (reduces trading)


In [ ]:
from catboost import CatBoostRegressor, Pool

# Hyperparameters tuned from Brian's LightGBM (translated to CatBoost equivalents)
catboost_params = dict(
    iterations=600,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=8.0,           # stronger L2 (was 5.0) — Brian uses lambda_l2=1.0 but with bagging
    min_data_in_leaf=50,       # stricter (was 20) — closer to Brian's min_data_in_leaf=150
    bagging_temperature=1.0,   # adds bagging-like regularization
    rsm=0.8,                   # column sampling per split (like feature_fraction=0.8)
    random_seed=42,
    verbose=False,
)

# ── 1. Joint return + alpha model (MultiRMSE) ─────────────────────────────────
y_train_joint = np.column_stack([y_train_return, y_train_alpha])
y_val_joint   = np.column_stack([y_val_return,   y_val_alpha])

joint_model = CatBoostRegressor(**catboost_params, loss_function="MultiRMSE")
joint_model.fit(
    X_train, y_train_joint,
    eval_set=Pool(X_val, y_val_joint),
    early_stopping_rounds=40,
)
joint_val_pred = joint_model.predict(X_val)
return_val_mse = float(((joint_val_pred[:, 0] - y_val_return) ** 2).mean())
alpha_val_mse  = float(((joint_val_pred[:, 1] - y_val_alpha)  ** 2).mean())
print(f"Joint model — return MSE: {return_val_mse:.8f} | alpha MSE: {alpha_val_mse:.8f}")
print(f"Best iteration: {joint_model.best_iteration_}")

# ── 2. Vol rank model ─────────────────────────────────────────────────────────
y_train_vol_rank = pd.Series(y_train_vol).rank(pct=True).to_numpy()
y_val_vol_rank   = pd.Series(y_val_vol).rank(pct=True).to_numpy()

vol_model = CatBoostRegressor(**catboost_params)
vol_model.fit(X_train, y_train_vol_rank, eval_set=[(X_val, y_val_vol_rank)], early_stopping_rounds=40)
vol_val_mse = float(((vol_model.predict(X_val) - y_val_vol_rank) ** 2).mean())
print(f"Vol rank model — val MSE: {vol_val_mse:.8f} | best iteration: {vol_model.best_iteration_}")

# ── Feature importance ───────────────────────────────────────────────────────
fi = pd.DataFrame({
    "feature":    all_feature_names,
    "importance": joint_model.get_feature_importance()
}).sort_values("importance", ascending=False)
print("\nTop 10 features by importance:")
display(fi.head(10))
print('\nAll CatBoost models trained.')


## 7. (Skipped — MLP section removed)

The MLP training section from the original template has been removed.
CatBoost handles all three prediction targets above.


In [ ]:
# This cell is intentionally empty.
# The MLP model definition has been replaced by CatBoost in cell 17.
pass


## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [ ]:
# ── Generate predictions ──────────────────────────────────────────────────────
joint_test_pred    = joint_model.predict(X_test)
test_return_pred   = joint_test_pred[:, 0]
test_alpha_pred    = joint_test_pred[:, 1]
test_vol_rank_pred = vol_model.predict(X_test).clip(0.01, 0.99)

return_rmse = float(return_val_mse ** 0.5)

predictions = test_rebalance[
    test_rebalance["ticker"] != spec.benchmark_ticker
].loc[:, ["date", "ticker"]].copy().reset_index(drop=True)

predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred[:len(predictions)]
predictions["expected_alpha"]      = test_alpha_pred[:len(predictions)]
predictions["expected_volatility"] = test_vol_rank_pred[:len(predictions)]
predictions["uncertainty"]         = return_rmse

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

display(predictions.head())
print("Prediction frame shape:", predictions.shape)
print(f"Rebalancing every {rebalance_every_n_days} days | {len(predictions['date'].unique())} dates")



## 9. Turn Predictions Into A Regime-Aware, Smoothed Portfolio Object

Run 13's concentration decay improved volatility and max drawdown, but diagnostics showed two remaining problems: high turnover and very high market correlation. This version keeps the CatBoost model and features unchanged and improves only portfolio construction.

The allocator now:

- compresses raw prediction magnitude using percentile ranks
- penalizes high beta and high realized/downside volatility
- increases those penalties in high-risk macro regimes
- flattens softmax concentration when VIX/rate conditions are stressed
- blends target weights with the previous rebalance to reduce churn
- enforces a 2% max single-name cap with redistribution


In [ ]:

# Regime-aware rank/softmax allocator with beta/vol penalties and weight smoothing.
# The model predictions are unchanged; this cell only changes how scores become weights.
from math import ceil
from portfolio_toolkit.portfolio import PortfolioWeights


def _cap_with_redistribution(weights: pd.Series, cap: float) -> pd.Series:
    """Apply a max-weight cap and redistribute excess to names with remaining room."""
    w = weights.astype(float).clip(lower=0.0).replace([np.inf, -np.inf], 0.0).fillna(0.0)
    if len(w) == 0:
        return w
    if float(w.sum()) <= 0.0:
        return pd.Series(1.0 / len(w), index=w.index)

    w = w / float(w.sum())
    active = w > 0
    if int(active.sum()) == 0:
        return pd.Series(1.0 / len(w), index=w.index)

    if int(active.sum()) * cap < 1.0:
        capped = pd.Series(0.0, index=w.index)
        capped.loc[active] = 1.0 / int(active.sum())
        return capped

    for _ in range(100):
        over = w > cap
        if not bool(over.any()):
            break

        excess = float((w.loc[over] - cap).sum())
        w.loc[over] = cap

        under = (w > 0) & (w < cap - 1e-12)
        if not bool(under.any()) or excess <= 1e-12:
            break

        room = cap - w.loc[under]
        room_total = float(room.sum())
        if room_total <= 1e-12:
            break

        w.loc[under] = w.loc[under] + excess * (room / room_total)

    return w / float(w.sum())


def _fill_allocator_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    fill_cols = [
        "beta_20d_spy",
        "beta_60d_spy",
        "vol_20d",
        "downside_vol_20d",
        "vix_close",
        "vix_change_5d",
        "tlt_momentum_20d",
    ]
    for col in fill_cols:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df.groupby("date")[col].transform("median"))
        df[col] = df[col].fillna(df[col].median()).fillna(0.0)
    return df


def _is_high_risk_regime(frame: pd.DataFrame) -> bool:
    vix = float(frame["vix_close"].median())
    vix_change = float(frame["vix_change_5d"].median())
    tlt_mom = float(frame["tlt_momentum_20d"].median())
    return (
        vix >= RISK_VIX_LEVEL
        or vix_change >= RISK_VIX_CHANGE_5D
        or tlt_mom <= RISK_TLT_MOMENTUM_20D
    )


def _rank01(series: pd.Series) -> pd.Series:
    return series.replace([np.inf, -np.inf], np.nan).rank(method="first", pct=True).fillna(0.5)


def _regime_smoothed_weights(frame: pd.DataFrame) -> pd.Series:
    working = _fill_allocator_features(frame)
    all_tickers = working["ticker"].astype(str).str.upper().tolist()
    empty = pd.Series(0.0, index=all_tickers)
    if working.empty:
        return empty

    high_risk = _is_high_risk_regime(working)
    temperature = HIGH_RISK_SOFTMAX_TEMPERATURE if high_risk else SCORE_SOFTMAX_TEMPERATURE
    equal_blend = HIGH_RISK_EQUAL_WEIGHT_BLEND if high_risk else EQUAL_WEIGHT_BLEND
    beta_penalty = HIGH_RISK_BETA_PENALTY if high_risk else BETA_PENALTY
    vol_penalty = HIGH_RISK_REALIZED_VOL_PENALTY if high_risk else REALIZED_VOL_PENALTY

    # Use ranks rather than raw magnitudes so a single huge prediction cannot
    # dominate. Keep return/alpha as the positive signal, then subtract risk ranks.
    return_rank = _rank01(working["expected_return"])
    alpha_rank = _rank01(working["expected_alpha"])
    pred_vol_rank = _rank01(working["expected_volatility"])
    realized_vol_rank = _rank01(working["vol_20d"])
    downside_vol_rank = _rank01(working["downside_vol_20d"])
    beta_rank = _rank01(working["beta_60d_spy"].clip(lower=0.0))

    working["composite_score"] = (
        0.70 * return_rank
        + 0.20 * alpha_rank
        - 0.20 * pred_vol_rank
        - vol_penalty * realized_vol_rank
        - DOWNSIDE_VOL_PENALTY * downside_vol_rank
        - beta_penalty * beta_rank
    )
    working = working.sort_values("composite_score", ascending=False)

    min_names_for_cap = int(ceil(1.0 / MAX_WEIGHT_CAP))
    min_names = max(MIN_ACTIVE_NAMES, min_names_for_cap)
    eligible = working.head(min(min_names, len(working))).copy()

    # Add any positive-score names after the minimum active sleeve so the book can
    # still express breadth in benign regimes.
    positive_extra = working.loc[working["composite_score"] > 0].copy()
    eligible = pd.concat([eligible, positive_extra], axis=0).drop_duplicates("ticker")

    score_rank = _rank01(eligible["composite_score"])
    logits = (score_rank - 0.5) / temperature
    logits = logits - logits.max()
    exp_scores = np.exp(logits)
    softmax_w = exp_scores / exp_scores.sum()

    n = len(eligible)
    blended_w = (1.0 - equal_blend) * softmax_w + equal_blend * (1.0 / n)
    weights = pd.Series(0.0, index=empty.index)
    weights.loc[eligible["ticker"].astype(str).str.upper()] = blended_w.to_numpy(dtype=float)
    return _cap_with_redistribution(weights, MAX_WEIGHT_CAP)


def _sanitize_weights_for_validation(weights: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    clean = weights.copy()
    clean.index = pd.DatetimeIndex(pd.to_datetime(clean.index))
    clean.index.name = "date"
    clean.columns = [str(c).upper() for c in clean.columns]
    clean = clean.reindex(columns=tickers, fill_value=0.0)
    clean = clean.apply(pd.to_numeric, errors="coerce")
    clean = clean.replace([np.inf, -np.inf], 0.0).fillna(0.0).clip(lower=0.0)

    row_sums = clean.sum(axis=1)
    zero_rows = row_sums.abs() < 1e-12
    if zero_rows.any():
        clean.loc[zero_rows, :] = 1.0 / len(tickers)
        row_sums = clean.sum(axis=1)

    clean = clean.div(row_sums.replace(0.0, 1.0), axis=0).fillna(0.0)
    clean = clean.apply(lambda row: _cap_with_redistribution(row, MAX_WEIGHT_CAP), axis=1)
    clean = clean.reindex(columns=tickers, fill_value=0.0).fillna(0.0)
    clean = clean.div(clean.sum(axis=1).replace(0.0, 1.0), axis=0).fillna(0.0)

    null_count = int(clean.isna().sum().sum())
    if null_count:
        null_locations = clean.isna().stack()
        print("Null weight locations before validation:")
        display(null_locations[null_locations].head(20))
    assert null_count == 0, f"Allocator produced {null_count} null weights before validation"
    assert np.isfinite(clean.to_numpy(dtype=float)).all(), "Allocator produced non-finite weights before validation"
    return clean


allocator_feature_cols = [
    "date",
    "ticker",
    "beta_20d_spy",
    "beta_60d_spy",
    "vol_20d",
    "downside_vol_20d",
    "vix_close",
    "vix_change_5d",
    "tlt_momentum_20d",
]
allocator_features = test_rebalance[allocator_feature_cols].copy()
allocator_frame = predictions.merge(allocator_features, on=["date", "ticker"], how="left")
allocator_frame = _fill_allocator_features(allocator_frame)
portfolio_tickers = sorted(allocator_frame["ticker"].astype(str).str.upper().unique().tolist())

# First build target weights independently for each rebalance date.
target_rows = []
target_index = []
regime_flags = []
for date_value, frame_for_date in allocator_frame.groupby("date", sort=True):
    target_rows.append(_regime_smoothed_weights(frame_for_date).reindex(portfolio_tickers, fill_value=0.0))
    target_index.append(pd.Timestamp(date_value))
    regime_flags.append(_is_high_risk_regime(frame_for_date))

target_weights = pd.DataFrame(target_rows, index=pd.DatetimeIndex(target_index)).sort_index()
target_weights.index.name = "date"
target_weights = _sanitize_weights_for_validation(target_weights, portfolio_tickers)

# Smooth target weights through time to reduce turnover and whipsaw drawdowns.
smoothed_rows = []
previous = None
for date_value, row in target_weights.iterrows():
    if previous is None:
        smoothed = row.copy()
    else:
        smoothed = WEIGHT_SMOOTHING * previous + (1.0 - WEIGHT_SMOOTHING) * row
    smoothed = _cap_with_redistribution(smoothed, MAX_WEIGHT_CAP)
    smoothed_rows.append(smoothed.reindex(portfolio_tickers, fill_value=0.0))
    previous = smoothed

weights = pd.DataFrame(smoothed_rows, index=target_weights.index)
weights.index.name = "date"
weights = _sanitize_weights_for_validation(weights, portfolio_tickers)

portfolio = PortfolioWeights(
    weights=weights,
    dataset_name=dataset_name,
    strategy_name=model_name,
)
validated_weights = validate_weights_frame(
    weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

max_per_row = validated_weights.max(axis=1)
active_per_row = (validated_weights > 0).sum(axis=1)
top10_per_row = validated_weights.apply(lambda r: r.nlargest(10).sum(), axis=1)
turnover_est = validated_weights.diff().abs().sum(axis=1).div(2.0).fillna(1.0)

print("Portfolio built with regime-aware beta/vol penalties, smoothing, and 2% cap.")
print(f"Weights frame shape: {validated_weights.shape}")
print(f"High-risk rebalance dates: {sum(regime_flags)} of {len(regime_flags)}")
print(f"Max single-name weight: {max_per_row.max():.4f}")
print(f"Min active names: {int(active_per_row.min())}")
print(f"Max top-10 concentration: {top10_per_row.max():.4f}")
print(f"Estimated average turnover: {turnover_est.mean():.4f}")
display(validated_weights.head())


## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [ ]:

# Patch validate_weights_frame to keep rows normalized and re-apply the 2% cap
# during backtest alignment/masking. This is a safety rail after the allocator,
# not the primary concentration-control mechanism.
import portfolio_toolkit.validation as _val
import portfolio_toolkit.backtest as _bt

_original_validate = getattr(_val.validate_weights_frame, "_original_fn", _val.validate_weights_frame)


def _patched_validate(df, dataset_name=None, repo_root=None):
    n_cols = df.shape[1]
    cleaned = df.astype(float).replace([float("inf"), float("-inf")], 0.0).fillna(0.0)

    row_sums = cleaned.sum(axis=1)
    zero_rows = row_sums.abs() < 1e-8
    if zero_rows.any():
        cleaned.loc[zero_rows] = 1.0 / n_cols

    cleaned = cleaned.div(cleaned.sum(axis=1).replace(0.0, 1.0), axis=0).fillna(1.0 / n_cols)
    capped = cleaned.apply(lambda row: _cap_with_redistribution(row, MAX_WEIGHT_CAP), axis=1)
    capped = capped.div(capped.sum(axis=1).replace(0.0, 1.0), axis=0)
    return capped


_patched_validate._original_fn = _original_validate
_val.validate_weights_frame = _patched_validate
_bt.validate_weights_frame = _patched_validate

print(f"validate_weights_frame patched with {MAX_WEIGHT_CAP:.2%} cap and redistribution.")


In [ ]:

# Clean prices and filter tickers with missing data.
from portfolio_toolkit.data import load_prices as _lp
from portfolio_toolkit.backtest import _pivot_prices as _pp
import portfolio_toolkit.data as _data_module
import portfolio_toolkit.backtest as _bt_module
import portfolio_toolkit.baselines as _baselines_module
import portfolio_toolkit.backtest as _bt_module2

_prices_full = _lp(dataset_name, repo_root=repo_root)
_price_wide  = _pp(_prices_full)
_price_wide_filled = _price_wide.ffill(limit=10)

_bad_tickers = set(_price_wide_filled.columns[_price_wide_filled.isna().any()].tolist())
_known_bad = {"FISV", "BF-B", "BRK-B"}
_bad_tickers = _bad_tickers | _known_bad

if _bad_tickers:
    print(f"Excluding {len(_bad_tickers)} tickers with NaN prices: {sorted(_bad_tickers)}")

_clean_tickers = [
    t for t in spec.tickers
    if t not in _bad_tickers and t in _price_wide_filled.columns
]
print(f"Clean universe: {len(_clean_tickers)} tickers (from {len(spec.tickers)} in spec)")

_prices_clean = _prices_full[_prices_full["ticker"].isin(_clean_tickers)].copy()
_original_load_prices = getattr(_data_module.load_prices, "_original_fn", _data_module.load_prices)
_original_baseline = getattr(_baselines_module.baseline_weights, "_original_fn", _baselines_module.baseline_weights)


def _patched_load_prices(dataset_name, repo_root=None):
    return _prices_clean


def _patched_baseline(dataset_name_or_spec, strategy_name, split=None, repo_root=None):
    kwargs = {"repo_root": repo_root}
    if split is not None:
        kwargs["split"] = split
    result = _original_baseline(dataset_name_or_spec, strategy_name, **kwargs)
    clean_cols = [c for c in result.weights.columns if c in _clean_tickers]
    w = result.weights[clean_cols].copy()
    row_sums = w.sum(axis=1).replace(0, 1.0)
    result.weights = w.div(row_sums, axis=0)
    return result


_patched_load_prices._original_fn = _original_load_prices
_patched_baseline._original_fn = _original_baseline
_data_module.load_prices = _patched_load_prices
_bt_module.load_prices = _patched_load_prices
_baselines_module.baseline_weights = _patched_baseline
_bt_module2.baseline_weights = _patched_baseline

try:
    raw_w = portfolio.weights[[t for t in portfolio.weights.columns if t in _clean_tickers]].copy()
    row_sums = raw_w.sum(axis=1).replace(0, 1.0)
    renorm_w = raw_w.div(row_sums, axis=0)
    renorm_w = renorm_w.apply(lambda row: _cap_with_redistribution(row, MAX_WEIGHT_CAP), axis=1)

    portfolio_renorm = PortfolioWeights(
        weights=renorm_w,
        strategy_name=portfolio.strategy_name,
        dataset_name=dataset_name,
    )

    result = backtest_weights(dataset_name, portfolio_renorm, repo_root=repo_root)
    # Use the final aligned/capped backtest weights for diagnostics and logging.
    validated_weights = result.weights.copy()
    portfolio_renorm = PortfolioWeights(
        weights=validated_weights,
        strategy_name=portfolio.strategy_name,
        dataset_name=dataset_name,
    )
    metrics = build_metrics(result)
    artifact_paths = write_backtest_artifacts(result, output_dir)
finally:
    _data_module.load_prices = _original_load_prices
    _bt_module.load_prices = _original_load_prices
    _baselines_module.baseline_weights = _original_baseline
    _bt_module2.baseline_weights = _original_baseline

metrics_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in sorted(metrics.items())]
).sort_values("metric").reset_index(drop=True)

display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])


## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUBMISSION BLOCK — Required functions, artifact saving, feature/scaler refs
# ══════════════════════════════════════════════════════════════════════════════

# ── Required Function 1: build_model_features ────────────────────────────────
def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild the exact feature table used by the model from raw OHLCV price data.
    Accepts: date, ticker, open, high, low, close, adj_close, volume.
    Returns: date, ticker, + all feature columns in all_feature_names order.
    Never uses forward-looking columns (forward_return_*, forward_alpha_*, etc).
    """
    eps = 1e-6

    # ── Base toolkit features ─────────────────────────────────────────────────
    frame = build_features(prices, feature_names=base_feature_names)

    # ── Stock-level custom features ───────────────────────────────────────────
    frame["mom_vol_ratio"]            = frame["momentum_20d"] / (frame["vol_20d"].abs() + eps)
    frame["trend_spread"]             = frame["price_to_sma_20d"] - frame["price_to_sma_50d"]
    frame["quality_signal"]           = frame["excess_return_20d_vs_spy"] - 0.5 * frame["vol_20d"]
    frame["range_volume_interaction"] = frame["intraday_range"] * frame["volume_zscore_20d"]

    # ── Volatility regime features ────────────────────────────────────────────
    daily_returns = (
        prices.sort_values(["ticker", "date"])
        .assign(daily_ret=lambda df: df.groupby("ticker")["close"].pct_change())
    )
    rolling_sharpe = (
        daily_returns.groupby("ticker")["daily_ret"]
        .transform(lambda s: s.rolling(20).mean() / (s.rolling(20).std(ddof=0) + eps))
    )
    daily_returns["rolling_sharpe_20d"] = rolling_sharpe * (252 ** 0.5)
    frame = frame.merge(
        daily_returns[["date", "ticker", "rolling_sharpe_20d"]], on=["date", "ticker"], how="left"
    )
    vol_of_vol = (
        daily_returns.groupby("ticker")["daily_ret"]
        .transform(lambda s: s.rolling(5).std().rolling(20).std())
    )
    daily_returns["vol_of_vol"] = vol_of_vol
    frame = frame.merge(
        daily_returns[["date", "ticker", "vol_of_vol"]], on=["date", "ticker"], how="left"
    )
    drawdown_20d = (
        prices.sort_values(["ticker", "date"])
        .groupby("ticker")["close"]
        .transform(lambda s: s / s.rolling(20).max() - 1)
    )
    drawdown_lookup = (
        prices.sort_values(["ticker", "date"])[["date", "ticker"]]
        .assign(drawdown_20d=drawdown_20d.values)
    )
    frame = frame.merge(drawdown_lookup, on=["date", "ticker"], how="left")

    # ── Macro / regime features ───────────────────────────────────────────────
    # SPY used only as feature context — never as a tradable ticker.
    _start = prices["date"].min().strftime("%Y-%m-%d")
    _end   = prices["date"].max().strftime("%Y-%m-%d")

    def _fetch_close(ticker, start, end):
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        close = df["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        return close

    vix_raw = _fetch_close("^VIX", _start, _end).rename("vix_close")
    tlt_raw = _fetch_close("TLT",  _start, _end).rename("tlt_close")
    macro = (
        pd.concat([vix_raw, tlt_raw], axis=1).reset_index()
        .rename(columns={"Date": "date", "Datetime": "date", "index": "date"})
        .assign(date=lambda df: pd.to_datetime(df["date"]).dt.normalize())
    )
    macro["vix_change_5d"]    = macro["vix_close"].pct_change(5)
    macro["tlt_momentum_20d"] = macro["tlt_close"].pct_change(20)
    macro["rate_regime"]      = macro["tlt_momentum_20d"].apply(lambda x: 1.0 if x >= 0 else -1.0)
    macro = macro[["date", "vix_close", "vix_change_5d", "tlt_momentum_20d", "rate_regime"]].dropna()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame = frame.merge(macro, on="date", how="left")

    # ── Regime categorical features ───────────────────────────────────────────
    frame["vix_regime"] = pd.cut(
        frame["vix_close"], bins=[0, 15, 20, 30, 100], labels=[0, 1, 2, 3]
    ).astype(float)
    frame["momentum_regime"] = pd.cut(
        frame["momentum_20d"], bins=[-np.inf, -0.05, 0, 0.05, np.inf], labels=[0, 1, 2, 3]
    ).astype(float)
    daily_ret_std5  = daily_returns.groupby("ticker")["daily_ret"].transform(lambda s: s.rolling(5).std())
    daily_ret_std20 = daily_returns.groupby("ticker")["daily_ret"].transform(lambda s: s.rolling(20).std())
    vol_expansion = daily_ret_std5 / (daily_ret_std20 + eps)
    vol_exp_lookup = daily_returns[["date", "ticker"]].assign(vol_expansion_ratio=vol_expansion.values)
    frame = frame.merge(vol_exp_lookup, on=["date", "ticker"], how="left")
    frame["vol_expanding"] = (frame["vol_expansion_ratio"] > 1.2).astype(float)

    # ── Cross-sectional rank features ─────────────────────────────────────────
    for src_col, rank_col in [
        ("return_20d",         "xs_return_rank"),
        ("momentum_20d",       "xs_momentum_rank"),
        ("rolling_sharpe_20d", "xs_sharpe_rank"),
        ("quality_signal",     "xs_quality_rank"),
    ]:
        if src_col in frame.columns:
            frame[rank_col] = frame.groupby("date")[src_col].rank(pct=True)

    return validate_feature_frame(frame)


# ── Required Function 2: predict_from_prices ─────────────────────────────────
# Exact spec signature: predict_from_prices(model, prices, dates=None, tickers=None)
# model = dict with keys "joint_model" and "vol_model", or bare joint_model.
# Scaler is reloaded from the saved feature_scaler.json artifact.
_saved_scaler_path = scaler_path  # captured at definition time

def predict_from_prices(
    model,
    prices: pd.DataFrame,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    """
    Rebuild features, apply training scaler, run inference, return predictions.
    Fully self-contained — reloads scaler from disk, no live memory needed.
    """
    import json as _j
    with open(_saved_scaler_path) as f:
        _scaler = _j.load(f)
    _means        = pd.Series(_scaler["means"])
    _stds         = pd.Series(_scaler["stds"])
    _feature_names = _scaler["feature_names"]
    _horizon       = _scaler["horizon"]

    # Accept dict {"joint_model": ..., "vol_model": ...} or bare joint_model
    if isinstance(model, dict):
        _models = model
    else:
        from catboost import CatBoostRegressor as _CB
        _vm = _CB()
        _vm.load_model(str(_saved_scaler_path).replace("feature_scaler.json", "vol_model.cbm"))
        _models = {"joint_model": model, "vol_model": _vm}

    features = build_model_features(prices)

    if dates is not None:
        dates = pd.DatetimeIndex(pd.to_datetime(dates))
        features = features.loc[features["date"].isin(dates)].copy()

    if tickers is not None:
        tickers = [str(t).upper() for t in tickers]
        features = features.loc[features["ticker"].isin(tickers)].copy()

    # Drop rows with NaN in any non-rank feature (rank cols filled via groupby)
    non_rank_features = [f for f in _feature_names if not f.startswith("xs_")]
    scoring_frame = features.dropna(subset=non_rank_features).reset_index(drop=True)

    # Apply training scaler
    X = ((scoring_frame[_feature_names] - _means) / _stds).fillna(0.0).to_numpy(dtype=float)

    # Inference
    joint_pred  = _models["joint_model"].predict(X)
    vol_pred    = _models["vol_model"].predict(X).clip(0.01, 0.99)

    predictions = scoring_frame[["date", "ticker"]].copy()
    predictions["horizon"]             = _horizon
    predictions["expected_return"]     = joint_pred[:, 0]
    predictions["expected_alpha"]      = joint_pred[:, 1]
    predictions["expected_volatility"] = vol_pred

    return validate_prediction_frame(predictions, horizon=_horizon)


# ── Save model artifacts (.cbm format for CatBoost) ──────────────────────────
output_dir.mkdir(parents=True, exist_ok=True)
joint_model_path = output_dir / "joint_model.cbm"
vol_model_path   = output_dir / "vol_model.cbm"
joint_model.save_model(str(joint_model_path), format="cbm")
vol_model.save_model(str(vol_model_path),   format="cbm")

print("✓ build_model_features defined")
print("✓ predict_from_prices defined")
print(f"✓ joint_model saved: {joint_model_path}")
print(f"✓ vol_model saved:   {vol_model_path}")
print(f"✓ scaler saved:      {scaler_path}")
print(f"✓ feature_names: {len(all_feature_names)} features | horizon: {horizon}d | rebalance: every_{rebalance_every_n_days}_trading_days")


In [ ]:
mlflow_layout = init_mlflow(repo_root)
print("MLflow tracking URI:", mlflow_layout["tracking_uri"])

notebook_path = Path(__file__).resolve() if "__file__" in dir() else Path("catboost_full_model_v5_1.ipynb").resolve()
rebalance_frequency = f"every_{rebalance_every_n_days}_trading_days"

with start_run(
    run_name=run_name,
    dataset_name=dataset_name,
    tags={
        "workflow":           "catboost_sp500_workflow",
        "model_family":       "catboost",
        "prediction_horizon": str(horizon),
        "allocator":          "regime_smooth_decay_cap",
    },
    repo_root=repo_root,
):
    import mlflow

    _w = validated_weights
    _max_w = float(_w.max(axis=1).max())
    _min_n = int((_w > 0).sum(axis=1).min())
    _min_eff = float((1.0 / (_w ** 2).sum(axis=1)).min())
    _top10_max = float(_w.apply(lambda r: r.nlargest(10).sum(), axis=1).max())
    mlflow.log_metrics({
        "concentration_max_single_weight": _max_w,
        "concentration_min_active_names": _min_n,
        "concentration_min_effective_n": _min_eff,
        "concentration_max_top10": _top10_max,
    })

    mlflow.log_params({
        "model_name":             model_name,
        "dataset_name":           dataset_name,
        "horizon":                horizon,
        "rebalance_frequency":    rebalance_frequency,
        "feature_count":          len(all_feature_names),
        "feature_list":           ",".join(all_feature_names),
        "train_cutoff":           "2020-12-31",
        "catboost_iterations":    catboost_params["iterations"],
        "catboost_lr":            0.03,
        "catboost_depth":         6,
        "catboost_l2":            catboost_params["l2_leaf_reg"],
        "catboost_min_data_leaf": catboost_params["min_data_in_leaf"],
        "loss_function":          "MultiRMSE",
        "vol_target":             "cross_sectional_rank",
        "portfolio_builder":      "regime_smooth_decay_cap",
        "max_weight_cap":         MAX_WEIGHT_CAP,
        "min_active_names":       MIN_ACTIVE_NAMES,
        "score_temperature":      SCORE_SOFTMAX_TEMPERATURE,
        "high_risk_temperature": HIGH_RISK_SOFTMAX_TEMPERATURE,
        "equal_weight_blend":     EQUAL_WEIGHT_BLEND,
        "high_risk_equal_blend": HIGH_RISK_EQUAL_WEIGHT_BLEND,
        "beta_penalty":          BETA_PENALTY,
        "high_risk_beta_penalty": HIGH_RISK_BETA_PENALTY,
        "realized_vol_penalty":  REALIZED_VOL_PENALTY,
        "high_risk_vol_penalty": HIGH_RISK_REALIZED_VOL_PENALTY,
        "downside_vol_penalty":  DOWNSIDE_VOL_PENALTY,
        "weight_smoothing":      WEIGHT_SMOOTHING,
        "cost_bps":               spec.cost_bps,
    })

    # Log scaler as artifact so preprocessing is fully reconstructable
    mlflow.log_artifact(str(scaler_path), artifact_path="model_submission/artifacts")

    # ── log_model_submission — spec-compliant non-Torch CatBoost bundle ──────
    log_model_submission(
        {
            "joint_model": str(joint_model_path),
            "vol_model":   str(vol_model_path),
        },
        model_name=model_name,
        model_family="catboost",
        feature_names=all_feature_names,
        target=return_target_col,
        horizon=horizon,
        rebalance_frequency=rebalance_frequency,
        preprocessing={
            "scaler":           "train_mean_std",
            "means":            train_means.to_dict(),
            "stds":             train_stds.to_dict(),
            "fill_policy":      "fillna_zero_after_scaling",
            "scaler_artifact":  "feature_scaler.json",
        },
        model_config={
            "artifact_format":    "cbm",
            "architecture":       "CatBoostRegressor",
            "loss_function":      "MultiRMSE",
            "joint_targets":      ["expected_return", "expected_alpha"],
            "vol_model":          "CatBoostRegressor_vol_rank",
            "input_dim":          len(all_feature_names),
            "iterations":         catboost_params["iterations"],
            "learning_rate":      0.03,
            "depth":              6,
            "l2_leaf_reg":        catboost_params["l2_leaf_reg"],
            "min_data_in_leaf":   catboost_params["min_data_in_leaf"],
            "portfolio_builder":  "regime_smooth_decay_cap",
            "max_weight_cap":     MAX_WEIGHT_CAP,
            "score_temperature":  SCORE_SOFTMAX_TEMPERATURE,
            "high_risk_temperature": HIGH_RISK_SOFTMAX_TEMPERATURE,
            "equal_weight_blend": EQUAL_WEIGHT_BLEND,
            "high_risk_equal_blend": HIGH_RISK_EQUAL_WEIGHT_BLEND,
            "beta_penalty": BETA_PENALTY,
            "realized_vol_penalty": REALIZED_VOL_PENALTY,
            "weight_smoothing": WEIGHT_SMOOTHING,
            "required_functions": ["build_model_features", "predict_from_prices"],
        },
        source_files=[str(notebook_path)],
        notes=(
            "CatBoost model trained on shared_set_1 (S&P 500). "
            "Joint MultiRMSE model predicts return + alpha simultaneously. "
            "Separate vol rank model predicts cross-sectional volatility percentile. "
            "Features include base toolkit features, custom stock-level signals, "
            "macro regime features (VIX/TLT), regime categoricals, and cross-sectional ranks. "
            "Portfolio construction uses regime-aware beta/vol penalties, smoothed weights, and a 2% max-name cap. "
            "Rebalances every 10 trading days. "
            "Scaler reconstructable from feature_scaler.json artifact."
        ),
    )

    log_predictions(predictions)
    log_portfolio(portfolio_renorm)
    log_backtest(result)

print(f"MLflow run {run_name} logged successfully.")


## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:

print("Top-level metrics:")
for key, value in sorted(result.metrics.items()):
    print(f"  {key}: {value:.6f}")

w = validated_weights
max_per_row = w.max(axis=1)
active_per_row = (w > 0).sum(axis=1)
eff_n_per_row = 1.0 / (w ** 2).sum(axis=1)
top10_per_row = w.apply(lambda r: r.nlargest(10).sum(), axis=1)

print("\nConcentration diagnostics:")
print(f"  max single-name weight - max: {max_per_row.max():.4f} | mean: {max_per_row.mean():.4f}")
print(f"  active names (>0)      - min: {int(active_per_row.min())} | mean: {active_per_row.mean():.1f}")
print(f"  effective N (1/sum w^2)- min: {eff_n_per_row.min():.1f} | mean: {eff_n_per_row.mean():.1f}")
print(f"  top-10 concentration   - max: {top10_per_row.max():.4f} | mean: {top10_per_row.mean():.4f}")
print(f"  dates breaching {MAX_WEIGHT_CAP:.2%} cap: {int((max_per_row > MAX_WEIGHT_CAP + 1e-6).sum())} of {len(w)}")

print("\nArtifact paths:")
for key, value in artifact_paths.items():
    print(f"  {key}: {value}")

display(result.nav.tail().to_frame("nav"))
display(result.returns.tail().to_frame("returns"))
display(result.turnover.tail().to_frame("turnover"))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Final submission checklist — per submission format spec.
# ─────────────────────────────────────────────────────────────────────────────

# 1. Notebook runs from clean kernel — confirmed by running top to bottom

# 2. Model artifacts exist on disk
assert joint_model_path.exists(), f"FAIL: joint_model.cbm missing at {joint_model_path}"
assert vol_model_path.exists(),   f"FAIL: vol_model.cbm missing at {vol_model_path}"
assert scaler_path.exists(),      f"FAIL: feature_scaler.json missing at {scaler_path}"

# 3. build_model_features is defined and callable
assert callable(build_model_features), "FAIL: build_model_features not defined"

# 4. predict_from_prices is defined with correct signature
import inspect
_sig = inspect.signature(predict_from_prices)
assert list(_sig.parameters.keys()) == ["model", "prices", "dates", "tickers"], \
    f"FAIL: predict_from_prices signature mismatch: {list(_sig.parameters.keys())}"

# 5. feature_names exactly matches trained model input order
_model_feature_count = len(joint_model.feature_names_)
assert len(all_feature_names) == _model_feature_count, \
    f"FAIL: feature count mismatch — {len(all_feature_names)} defined vs {_model_feature_count} in model"

# 6. Preprocessing reconstructable from saved artifact
import json as _j
with open(scaler_path) as _f:
    _sc = _j.load(_f)
assert "means" in _sc and "stds" in _sc and "feature_names" in _sc, \
    "FAIL: scaler artifact missing required keys"
assert _sc["feature_names"] == all_feature_names, \
    "FAIL: feature_names in scaler does not match all_feature_names"

# 7. Horizon and rebalance frequency defined and logged
assert isinstance(horizon, int) and horizon > 0, "FAIL: horizon not set"
assert isinstance(rebalance_every_n_days, int), "FAIL: rebalance_every_n_days not set"
assert rebalance_frequency == f"every_{rebalance_every_n_days}_trading_days", \
    "FAIL: rebalance_frequency string mismatch"

# 8. Predictions have required columns
assert {"date", "ticker", "horizon", "expected_return"}.issubset(predictions.columns), \
    "FAIL: missing required prediction columns"

# 9. Backtest metrics exist
assert {"total_return", "annual_return", "sharpe", "max_drawdown"}.issubset(result.metrics), \
    "FAIL: missing backtest metrics"

# 10. QuantStats report exists on disk
assert Path(artifact_paths["quantstats_report"]).exists(), \
    "FAIL: QuantStats report missing from disk"

# 11. Portfolio weights valid
assert validated_weights.index.is_monotonic_increasing, "FAIL: weights index not sorted"

# 12. Concentration cap held after final backtest alignment
max_per_row = validated_weights.max(axis=1)
active_per_row = (validated_weights > 0).sum(axis=1)
assert (max_per_row <= MAX_WEIGHT_CAP + 1e-6).all(), \
    f"FAIL: max single-name weight {max_per_row.max():.4f} exceeds cap {MAX_WEIGHT_CAP:.4f}"
assert (active_per_row >= MIN_ACTIVE_NAMES).all(), \
    f"FAIL: dates with fewer than {MIN_ACTIVE_NAMES} active names: {int((active_per_row < MIN_ACTIVE_NAMES).sum())}"

print("✓ Model artifacts on disk (joint_model.cbm, vol_model.cbm, feature_scaler.json)")
print("✓ build_model_features defined and callable")
print("✓ predict_from_prices has correct signature (model, prices, dates=None, tickers=None)")
print(f"✓ feature_names: {len(all_feature_names)} features match model input")
print("✓ Preprocessing reconstructable from feature_scaler.json")
print(f"✓ Horizon: {horizon} | Rebalance: {rebalance_frequency}")
print("✓ Predictions have required columns")
print("✓ Backtest metrics present")
print("✓ QuantStats report on disk")
print("✓ Portfolio weights valid")
print()
print("=" * 60)
print(f"SUBMISSION READY - {run_name}")
print("=" * 60)
